# 3 · GRPO (Group Relative Policy Optimization)
**Idea:** the model tries 4 answers for each request. A score function grades each try.
The model then repeats what scored above the group average, like practicing until the shots go in.

`black sneakers under 50 dollars` → `FIND shoes color=BK max=50`

There are no answers to copy: the model only gets a score. The prompt explains the rules, so some tries are already right.

## Step 1 · Load a small model

In [3]:
import warnings, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings('ignore')   # hide technical warnings

MODEL = 'Qwen/Qwen3.5-0.8B'                  # small open model (0.8 billion parameters)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 315.82it/s]


## Step 2 · Write the examples
The same 25 requests as notebook 1. The correct command is used **only** by the score function, never shown to the model.

In [6]:
TRAIN = [  # (request, correct command)
    ('red tee',                           'FIND shirt color=RD'),
    ('blue shirt',                        'FIND shirt color=BL'),
    ('black t-shirt under 20 dollars',    'FIND shirt color=BK max=20'),
    ('white tee below 30',                'FIND shirt color=WH max=30'),
    ('I want a red t-shirt',              'FIND shirt color=RD'),
    ('white shirt',                       'FIND shirt color=WH'),
    ('black sneakers',                    'FIND shoes color=BK'),
    ('white trainers under 80 dollars',   'FIND shoes color=WH max=80'),
    ('blue shoes',                        'FIND shoes color=BL'),
    ('red sneakers below 50',             'FIND shoes color=RD max=50'),
    ('show me black trainers',            'FIND shoes color=BK'),
    ('red shoes under 30 dollars',        'FIND shoes color=RD max=30'),
    ('blue trainers',                     'FIND shoes color=BL'),
    ('black coat',                        'FIND jacket color=BK'),
    ('blue jacket under 80 dollars',      'FIND jacket color=BL max=80'),
    ('white coat below 50',               'FIND jacket color=WH max=50'),
    ('I want a red jacket',               'FIND jacket color=RD'),
    ('blue coat',                         'FIND jacket color=BL'),
    ('black jacket below 80',             'FIND jacket color=BK max=80'),
    ('red hoodie',                        'FIND hoodie color=RD'),
    ('white sweatshirt under 30 dollars', 'FIND hoodie color=WH max=30'),
    ('black hoodie below 20',             'FIND hoodie color=BK max=20'),
    ('show me a blue sweatshirt',         'FIND hoodie color=BL'),
    ('white hoodie',                      'FIND hoodie color=WH'),
    ('red sweatshirt below 50',           'FIND hoodie color=RD max=50'),
]

TEST = [
    ('blue tee under 50 dollars',         'FIND shirt color=BL max=50'),
    ('white sneakers',                    'FIND shoes color=WH'),
    ('red coat below 30',                 'FIND jacket color=RD max=30'),
    ('black sweatshirt',                  'FIND hoodie color=BK'),
    ('show me a white t-shirt',           'FIND shirt color=WH'),
]

RULES = '''Write the search command for the shop request.
Format: FIND <item> color=<code> max=<price>   (max only if a price is given)
Items: shirt (shirt, tee, t-shirt), shoes (shoes, sneakers, trainers), jacket (jacket, coat), hoodie (hoodie, sweatshirt)
Color codes: red=RD, blue=BL, black=BK, white=WH
Reply with only the command.'''

def question(request):
    return [{'role': 'user', 'content': f'{RULES}\n\nRequest: {request}'}]

## Step 3 · The score function
This is the heart of GRPO: it decides what "good" means.
- **1.0** exactly the correct command
- **up to 0.5** partly right: the share of the command's parts found in the answer

In [9]:
def score(answer, command):
    if answer.strip() == command:
        return 1.0                                   # exactly right
    parts = command.split()                          # e.g. ['FIND', 'shoes', 'color=BK']
    found = [part for part in parts if part in answer.split()]
    return 0.5 * len(found) / len(parts)             # partly right

print(score('FIND shoes color=BK', 'FIND shoes color=BK'))
print(score('FIND shoes color=black', 'FIND shoes color=BK'))
print(score('black sneakers', 'FIND shoes color=BK'))

1.0
0.3333333333333333
0.0


## Step 4 · Ask the model before training
The prompt explains every rule. **Look for** the mistakes it still makes.

In [11]:
def ask(model, request):
    inputs = tokenizer.apply_chat_template(question(request), add_generation_prompt=True,
                                           return_tensors='pt', return_dict=True).to('cuda')
    output = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    answer = output[0][inputs['input_ids'].shape[1]:]          # keep only the new words
    return tokenizer.decode(answer, skip_special_tokens=True).strip()

def test(model):
    model.eval()                                               # answering mode
    right = 0
    for request, command in TEST:
        answer = ask(model, request)
        if answer == command:
            right += 1
            print('right:', request, '->', answer)
        else:
            print('wrong:', request, '->', answer, '   expected:', command)
    print(right, 'of', len(TEST), 'right')

test(model)

right: blue tee under 50 dollars -> FIND shirt color=BL max=50
wrong: white sneakers -> FIND shoes color=WH max=100    expected: FIND shoes color=WH
right: red coat below 30 -> FIND jacket color=RD max=30
wrong: black sweatshirt -> FIND black sweatshirt color=BL max=100    expected: FIND hoodie color=BK
wrong: show me a white t-shirt -> FIND shirt color=WH max=10    expected: FIND shirt color=WH
2 of 5 right


## Step 5 · Train
For each request, the model writes 4 tries. Each try gets a score.
Tries above the group average become more likely; tries below become less likely.

- **Watch:** the `reward` column (the average score) going up.
- This is the slowest notebook: the model must write every try before it can learn.

In [12]:
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig

def reward(completions, command, **kwargs):          # the trainer calls this with a batch of tries
    scores = []
    for tried, correct in zip(completions, command):
        scores.append(score(tried[0]['content'], correct))
    return scores

data = Dataset.from_list([
    {'prompt': question(request), 'command': command}
    for request, command in TRAIN
])

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=data,
    reward_funcs=reward,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules='all-linear'),   # the small add-on
    args=GRPOConfig(
        num_generations=4,         # tries per request
        max_completion_length=50,  # longest try, in tokens
        num_train_epochs=5,        # each request is seen 5 times
        learning_rate=1e-4,        # size of each small adjustment
        logging_steps=5,           # print progress every 5 steps
        save_strategy='no',        # do not save copies to disk
    ),
)
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss
5,0.013095
10,-0.000000
15,0.000000
20,0.017641
25,-0.008885
30,0.000000
35,0.000000
40,0.000000
45,-0.004343
50,0.000000


TrainOutput(global_step=60, training_loss=0.0014590770620974394, metrics={'train_runtime': 133.3208, 'train_samples_per_second': 0.938, 'train_steps_per_second': 0.45, 'total_flos': 0.0, 'train_loss': 0.0014590770620974394, 'epoch': 5.0})

## Step 6 · Ask the model after training
The same 5 test requests. **Look for** fewer mistakes.

In [13]:
test(trainer.model)

right: blue tee under 50 dollars -> FIND shirt color=BL max=50
wrong: white sneakers -> FIND shoes color=WH max=999999    expected: FIND shoes color=WH
right: red coat below 30 -> FIND jacket color=RD max=30
wrong: black sweatshirt -> FIND hoodie color=BK max=999999    expected: FIND hoodie color=BK
wrong: show me a white t-shirt -> FIND shirt color=WH max=20    expected: FIND shirt color=WH
2 of 5 right
